# Setup: Prerequisites for Korean LLM Evaluation

This notebook configures the required RBAC permissions and secrets to run `LMEvalJob` against an OAuth-protected KServe InferenceService on OpenShift AI.

## What is LMEvalJob?

LMEvalJob is a Custom Resource provided by the TrustyAI Operator that runs [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) as a Kubernetes Job. It handles dataset download, model inference, and result collection.

## Prerequisites

- OpenShift AI cluster with TrustyAI Operator installed
- A model deployed via KServe with `security.opendatahub.io/enable-auth: "true"`
- `oc` CLI logged in to the cluster
- Hugging Face token for gated model access

## Step 1: Set Your Configuration

Configuration is loaded from `../.env`. Copy `sample.env` to `.env` and update values before running:

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
HF_TOKEN = os.getenv("HF_TOKEN", "hf_xxxxx")
HF_TOKEN_SECRET = os.getenv("HF_TOKEN_SECRET", "hf-token")

print(f"Namespace: {NAMESPACE}")
print(f"HF Token: {HF_TOKEN[:8]}...")
print(f"HF Token Secret: {HF_TOKEN_SECRET}")

Namespace: demo
HF Token: hf_WsmWd...
HF Token Secret: hf-token


## Step 2: Verify Cluster Access

Make sure you are logged in to the OpenShift cluster. If not, run:

```bash
oc login --server=<your-cluster-api-url>
```

In [6]:
!oc whoami
!oc project {NAMESPACE}

kube:admin
Now using project "demo" on server "https://api.openshift-cluster.sandbox1785.opentlc.com:6443".


## Step 3: Identify Your Model

Find the InferenceService name — this is also the `served_model_name` used by vLLM:

In [7]:
!oc get inferenceservice -n {NAMESPACE}

NAME         URL                                                                      READY   PREV   LATEST   PREVROLLEDOUTREVISION   LATESTREADYREVISION   AGE
gemma4-e2b   https://gemma4-e2b-demo.apps.openshift-cluster.sandbox1785.opentlc.com   True                                                                  38h


## Step 4: Create RBAC Permissions

The LMEvalJob Pod uses the `default` ServiceAccount. When the InferenceService has OAuth enabled, the SA needs permission to `get` InferenceServices for the OAuth proxy to validate access.

In [8]:
!oc apply -f samples/role.yaml -n {NAMESPACE}
!oc apply -f samples/rolebinding.yaml -n {NAMESPACE}

role.rbac.authorization.k8s.io/inferenceservice-reader created
rolebinding.rbac.authorization.k8s.io/lmeval-inferenceservice-access created


## Step 5: Create Hugging Face Token Secret

Required for downloading gated tokenizers (e.g., `google/gemma-2b`):

In [9]:
!oc create secret generic {HF_TOKEN_SECRET} \
    --from-literal=HF_TOKEN={HF_TOKEN} \
    -n {NAMESPACE} \
    --dry-run=client -o yaml | oc apply -f -

secret/hf-token configured


## Step 6: Create Service Account Token Secret

This long-lived SA token is injected as `OPENAI_API_KEY` to authenticate with the OAuth-protected model endpoint:

In [10]:
!oc apply -f samples/sa-token-secret.yaml -n {NAMESPACE}

secret/lmeval-sa-token created


## Step 7: Verify Permissions

In [11]:
!oc auth can-i get inferenceservices.serving.kserve.io \
    -n {NAMESPACE} \
    --as=system:serviceaccount:{NAMESPACE}:default

yes


Expected output: `yes`

## Done!

You're now ready to run evaluations. Proceed to:
- **0_setup/2_eval_hub_setup.ipynb** to configure EvalHub SDK with MLflow experiment tracking
- **1_builtin_tasks/1_builtin_task_eval.ipynb** for a quick evaluation using built-in tasks
- **2_custom_tasks/1_custom_task_eval.ipynb** for full flexibility with Git-sourced tasks